# Neha reaction clips with LivePortrait (Colab T4)

Turns the still portrait `face.jpg` into real head / eye / lip motion by **transferring the motion** of short driving clips
(the presenter's reaction loops from `tran_moment/prepared/clips/`). Output: seven 406x720 loops with the same names the
card already understands (`idle, small_nod, slow_nod, look_up, both_hands, open_palm, greeting`) + `manifest.json`.

**Runtime -> Change runtime type -> T4 GPU**, then run the cells in order. ~10-15 min.

Inputs: upload `liveportrait_inputs.zip` (made by `tran_moment/path_c/make_inputs.ps1` / already in `path_c/`) when cell 3 asks.

Rights: the driving clips carry a staff presenter's *movements* only; the rendered face is the CGI Neha portrait. Keep the
licence check for the AI portrait (CONSENT.md). LivePortrait is MIT; the pretrained weights have their own licence - read
`pretrained_weights/README` before commercial use.

In [ ]:
# 1. GPU check
import torch, subprocess
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)

In [ ]:
# 2. LivePortrait + weights (first run ~5 min)
%cd /content
!git clone -q https://github.com/KwaiVGI/LivePortrait.git 2>/dev/null || true
%cd /content/LivePortrait
!pip install -q -r requirements.txt
!pip install -q 'huggingface_hub[cli]'
!huggingface-cli download KwaiVGI/LivePortrait --local-dir pretrained_weights --exclude '*.git*' 'README.md' 'docs' -q
!ls pretrained_weights/liveportrait

In [ ]:
# 3. Inputs: face.jpg + driving clips (upload liveportrait_inputs.zip)
import os, zipfile, glob
from google.colab import files
os.makedirs('/content/inputs', exist_ok=True)
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('/content/inputs')
    else:
        os.replace(name, f'/content/inputs/{name}')
SRC = '/content/inputs/face.jpg'
DRIVING = sorted(glob.glob('/content/inputs/clips/*.mp4'))
assert os.path.exists(SRC), 'face.jpg missing'
print(len(DRIVING), 'driving clips:', [os.path.basename(d) for d in DRIVING])

In [ ]:
# 4. Render: one LivePortrait pass per driving clip (~40-90 s each on T4)
import subprocess, os, shutil
OUT = '/content/out'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
for d in DRIVING:
    name = os.path.splitext(os.path.basename(d))[0]
    print('->', name)
    r = subprocess.run(['python', 'inference.py', '-s', SRC, '-d', d, '-o', f'{OUT}/{name}',
                        '--flag_crop_driving_video',      # driving clips are full portraits, let it find the face
                        '--flag_stitching',               # paste the animated face back into the full portrait
                        '--flag_relative_motion'],       # motion relative to the first frame -> clean loops
                       capture_output=True, text=True)
    if r.returncode: print(r.stdout[-800:], r.stderr[-800:])
print(sorted(os.listdir(OUT)))

In [ ]:
# 5. Package: pick the plain result (not the *_concat preview), crop to 9:16 -> 406x720, write manifest, zip
import glob, json, os, subprocess, shutil
PKG = '/content/clips_neha_render'; shutil.rmtree(PKG, ignore_errors=True); os.makedirs(PKG)
for d in DRIVING:
    name = os.path.splitext(os.path.basename(d))[0]
    cands = [f for f in glob.glob(f'{OUT}/{name}/*.mp4') if '_concat' not in f]
    if not cands: print('no output for', name); continue
    # source is 640x852 (3:4); crop a centred 9:16 window biased to the top (face), scale to the card size
    vf = "crop='min(iw,ih*9/16)':'min(ih,iw*16/9)':(iw-ow)/2:0,scale=406:720:flags=lanczos,format=yuv420p"
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', cands[0], '-an', '-vf', vf, '-c:v', 'libx264', '-preset', 'slow', '-crf', '20',
                    '-movflags', '+faststart', f'{PKG}/{name}.mp4'], check=True)
manifest = {
  '_note': 'LivePortrait render of prepared/face.jpg driven by the presenter reaction clips. Same keys as clips_neha/manifest.json.',
  'version': 1, 'set': 'neha_render', 'crossfade_ms': 350,
  'states': {'idle': ['idle.mp4'], 'greeting': ['greeting.mp4'], 'listening': ['small_nod.mp4'], 'thinking': ['look_up.mp4']},
  'gestures': {'idle': ['idle.mp4'], 'pause': ['idle.mp4'], 'small_nod': ['small_nod.mp4'], 'nod': ['small_nod.mp4', 'open_palm.mp4'],
               'slow_nod': ['slow_nod.mp4'], 'tilt_head': ['slow_nod.mp4'], 'open_palm': ['open_palm.mp4'], 'both_hands': ['both_hands.mp4'],
               'lean_in': ['both_hands.mp4', 'open_palm.mp4'], 'look_up': ['look_up.mp4']}}
json.dump(manifest, open(f'{PKG}/manifest.json', 'w'), indent=2)
shutil.make_archive('/content/clips_neha_render', 'zip', PKG)
print(sorted(os.listdir(PKG)))
from google.colab import files; files.download('/content/clips_neha_render.zip')

## Back on the laptop

Unzip into `tran_moment/prepared/clips_neha_render/` (so `manifest.json` and the seven mp4s sit directly inside), then open
`http://127.0.0.1:4003/clips?set=neha_render` to review, and `http://127.0.0.1:4003/embed?clips=neha_render` for the card.
To make it the default embed set: `data-clips="neha_render"` on the widget tag.

What this gives: real head/eye/lip motion in idle, listening, thinking and gesture states. What it does not: lip-sync to the live
TTS (that needs a real-time renderer - MuseTalk/LiveTalking on a GPU host, or a Tavus/HeyGen/D-ID stream).